# Ignite-3B S05 - Cond C gen 6-8 final (resume S04)

Ends Cond C. Push v_8 as `iterate-labs-ai/ignite-3b-v1`.

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
HF_REPO_INTERMEDIATE = 'iterate-labs-ai/ignite-3b-cond-c'
HF_REPO_FINAL = 'iterate-labs-ai/ignite-3b-v1'
PREV_KAGGLE = 'vitorscrt/ignite-3b-cond-c-s04'
OUT_ROOT = '/kaggle/working/cond_C_run'
GENS_TOTAL = 9
CANDS = 8
STEPS = 150

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'unsloth>=2025.1.0' 'trl>=0.12.0' 'vllm>=0.6.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'scipy' 'huggingface_hub' kaggle

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
import subprocess
subprocess.run(['kaggle', 'datasets', 'download', '-d', PREV_KAGGLE, '-p', OUT_ROOT, '--unzip', '--force'], check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'train.ignite.C_rsi_outer',
       '--base', BASE_MODEL,
       '--dataset-train', 'data/ignite/omni_math_train.jsonl',
       '--dataset-dev', 'data/ignite/omni_math_dev.jsonl',
       '--dataset-val', 'data/ignite/omni_math_val.jsonl',
       '--bench', 'math', '--bench-name', 'omni_math',
       '--gens', str(GENS_TOTAL), '--cands', str(CANDS), '--steps', str(STEPS),
       '--out', OUT_ROOT, '--hf-repo', HF_REPO_INTERMEDIATE, '--resume'], check=True)

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

log = [json.loads(l) for l in open(f'{OUT_ROOT}/log.jsonl')]
gens = [r for r in log if r.get('type') == 'generation']
if not gens:
    raise SystemExit('no accepted generations found')
last = gens[-1]
print(f'last gen={last["gen"]} val_r={last["val_r"]:.4f}')

api = HfApi()
api.create_repo(HF_REPO_FINAL, repo_type='model', exist_ok=True, private=True)
api.upload_folder(folder_path=last['v_ckpt'], repo_id=HF_REPO_FINAL, repo_type='model')
print(f'-> https://huggingface.co/{HF_REPO_FINAL}')